# korea_fs_data_from_DG 조회 · 분석 노트북 (V1)

`dataguide_fs_loader` 로 적재된 DataGuide 재무데이터를 **조회 / 추출 / 분석 / 시각화** 한다.
적재(collection) 노트북과 분리된 query 전용 노트북이며, 단독으로 top-to-bottom 실행된다.

## 실행 순서
| Cell | 내용 |
|---|---|
| 1 | **사용자 설정** — 이 셀의 변수만 수정 |
| 2 | 환경 준비 (경로 자동탐색 · DB 엔진 · 한글폰트) |
| 3 | 테이블 스키마 점검 · 인덱스 확인 |
| 4 | 재무항목(finance item) 마스터 구축 · 검색 함수 |
| 5 | ▶ 항목 검색 실행 |
| 6 | 재무데이터 추출 함수 |
| 7 | ▶ 데이터 추출 실행 |
| 8 | 기본 분석 (YoY · QoQ · TTM · CAGR) |
| 9 | 시각화 함수 |
| 10 | ▶ 시각화 실행 |
| 11 | 엑셀 저장 (옵션) |

## 테이블 구조 (loader 기준)
- PK = `(date, ticker, item_code)`
- 컬럼 = `date, ticker, company_name, item_code, indicator, value, market, sj_div, created_at, updated_at`
- `value` 원본 단위 = **천원**

## 주의
- **비금액 항목 자동 판별**: `%`, `률`, `배`, `주당`, `EPS/BPS` 등이 포함된 항목은 단위 환산에서 제외된다.
  전 항목에 일괄 `/1e5` 를 적용하면 ROE·PER 같은 비율 지표가 망가진다.
- 성장률 lag 은 데이터 주기에서 자동 판정한다 (분기=4, 연간=1).

In [8]:
# ==========================================================
# Cell 1: 사용자 설정  ── 이 셀의 변수만 수정하면 됩니다
# ==========================================================

# ---------- DB 접속 ----------
DB_PORT     = 3307                       # MariaDB 포트
DB_USER     = 'stox7412'                 # DB 사용자
DB_PASSWORD = 'Apt106503!~'              # DB 비밀번호
DB_NAME     = 'investar'                 # DB 스키마명
TABLE_NAME  = 'korea_fs_data_from_DG'    # 재무데이터 테이블명

# ---------- 조회 대상 ----------
TICKERS    = ['A278470', 'A003350']      # 티커 리스트 ('A'+6자리). None = 전체 (대용량 주의)
ITEM_NAMES = ['매출액(천원)', '영업이익(천원)']        # 재무항목명 리스트(정확명칭). None이면 ITEM_CODES 사용
ITEM_CODES = None                        # 항목코드 직접지정 예: ['M000904001']. ITEM_NAMES보다 우선
START_DATE = '2015-01-01'                # 조회 시작일 (YYYY-MM-DD)
END_DATE   = '2026-12-31'                # 조회 종료일 (YYYY-MM-DD)
MARKET     = None                        # 'KS' / 'KQ' / None(전체)
SJ_DIV     = None                        # 재무제표 구분 필터 / None(전체)

# ---------- 단위 · 분석 ----------
UNIT        = '억원'                      # 금액 환산: '천원'(원본) / '백만원' / '억원' / '조원'
FREQ        = 'auto'                     # 데이터 주기: 'auto' / 'Q'(분기) / 'A'(연간)
GROWTH_LAG  = None                       # 성장률 lag 수동지정. None이면 자동 (Q=4, A=1)
ADD_TTM     = True                       # True면 분기 유량항목에 TTM(4분기 누적) 컬럼 추가

# ---------- 항목 검색 (Cell 5) ----------
SEARCH_KEYWORD      = '매출'              # 항목명 검색 키워드 (부분일치)
REFRESH_ITEM_MASTER = False              # True면 항목 마스터 캐시를 DB에서 재생성
ITEM_MASTER_TOPN    = 60                 # 검색 결과 표시 개수

# ---------- 출력 ----------
SAVE_EXCEL     = False                   # True면 결과를 엑셀로 저장
CACHE_DIR_NAME = 'cache'                 # DATA 하위 캐시 폴더명
FIG_WIDTH      = 15                      # 그래프 가로 크기

In [9]:
# ==========================================================
# Cell 2: 환경 준비 (경로 자동탐색 · DB 엔진 · 한글폰트)
# ==========================================================
import sys, os, warnings
from pathlib import Path
from typing import Optional, List, Dict, Union, Tuple   # Python 3.9 : PEP604 미사용

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager, rc
from sqlalchemy import create_engine, text

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')


# ---- 1. 프로젝트 루트 자동 탐색 (DATA 폴더 기준 · 데스크탑/노트북 공용) ----
def add_repo_path() -> Path:
    """현재 경로에서 위로 올라가며 DATA 폴더를 가진 디렉터리를 프로젝트 루트로 잡는다."""
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / 'DATA').exists():
            if str(parent) not in sys.path:
                sys.path.insert(0, str(parent))
            return parent
    raise FileNotFoundError('DATA 폴더를 찾을 수 없습니다. 노트북 위치를 확인하세요.')


PROJECT_ROOT = add_repo_path()
DATA_DIR     = PROJECT_ROOT / 'DATA'
CACHE_DIR    = DATA_DIR / CACHE_DIR_NAME
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Cache dir    : {CACHE_DIR}')


# ---- 2. DB 접속정보 (loader 와 동일 규약) ----
from DATA.stock_invest_function import get_db_host

DB_INFO = {
    'host':     get_db_host(),
    'port':     DB_PORT,
    'user':     DB_USER,
    'password': DB_PASSWORD,
    'database': DB_NAME,
}
print(f"DB host      : {DB_INFO['host']}:{DB_INFO['port']}/{DB_INFO['database']}")


def build_engine(db_info: Dict):
    """공용 get_engine() 이 있으면 재사용, 없으면 직접 생성."""
    try:
        from DATA.stock_invest_function import get_engine
        eng = get_engine()
        if eng is not None:
            return eng
    except Exception:
        pass
    from urllib.parse import quote_plus
    url = (
        f"mysql+pymysql://{db_info['user']}:{quote_plus(db_info['password'])}"
        f"@{db_info['host']}:{db_info['port']}/{db_info['database']}?charset=utf8mb4"
    )
    return create_engine(url, pool_pre_ping=True, pool_recycle=3600)


ENGINE = build_engine(DB_INFO)

with ENGINE.connect() as conn:
    ver = conn.execute(text('SELECT VERSION()')).scalar()
print(f'DB version   : {ver}')


# ---- 3. 한글 폰트 (Windows/Mac/Linux 자동) ----
def set_korean_font() -> str:
    installed = {f.name for f in font_manager.fontManager.ttflist}
    for cand in ['Malgun Gothic', 'AppleGothic', 'NanumGothic',
                 'Noto Sans CJK KR', 'NanumBarunGothic', 'DejaVu Sans']:
        if cand in installed:
            rc('font', family=cand)
            matplotlib.rcParams['axes.unicode_minus'] = False
            return cand
    matplotlib.rcParams['axes.unicode_minus'] = False
    return '(기본 폰트 · 한글 깨질 수 있음)'


FONT_USED = set_korean_font()
print(f'Matplotlib font: {FONT_USED}')


# ---- 4. 단위 환산 테이블 (원본 = 천원) ----
UNIT_DIVISOR = {'천원': 1.0, '백만원': 1e3, '억원': 1e5, '조원': 1e9}
if UNIT not in UNIT_DIVISOR:
    raise ValueError(f'UNIT 은 {list(UNIT_DIVISOR)} 중 하나여야 합니다. 입력값={UNIT}')
print(f'단위 환산     : 천원 → {UNIT} (÷{UNIT_DIVISOR[UNIT]:,.0f})')

Project root : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
Cache dir    : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\DATA\cache
DB host      : 192.168.0.230:3307/investar
DB version   : 10.11.6-MariaDB
Matplotlib font: Malgun Gothic
단위 환산     : 천원 → 억원 (÷100,000)


In [10]:
# ==========================================================
# Cell 3: 테이블 스키마 점검 · 인덱스 확인
# ==========================================================
# 아래 결과로 실제 컬럼명을 먼저 확인한다. 이후 셀은 여기 컬럼명을 전제로 동작한다.

with ENGINE.connect() as conn:
    df_cols = pd.read_sql(text(f'SHOW COLUMNS FROM {TABLE_NAME}'), conn)
    df_idx  = pd.read_sql(text(f'SHOW INDEX FROM {TABLE_NAME}'), conn)

print('=' * 78)
print(f'[스키마] {TABLE_NAME}')
print('=' * 78)
print(df_cols.to_string(index=False))

print('\n' + '=' * 78)
print('[인덱스]')
print('=' * 78)
print(df_idx[['Key_name', 'Seq_in_index', 'Column_name', 'Non_unique']].to_string(index=False))

TABLE_COLUMNS = df_cols['Field'].tolist()
EXISTING_INDEX = set(df_idx['Key_name'].unique())

# ---- 필수 컬럼 확인 ----
REQUIRED = ['date', 'ticker', 'item_code', 'indicator', 'value']
missing = [c for c in REQUIRED if c not in TABLE_COLUMNS]
if missing:
    raise KeyError(f'필수 컬럼 누락: {missing}')

HAS_COMPANY = 'company_name' in TABLE_COLUMNS
HAS_MARKET  = 'market' in TABLE_COLUMNS
HAS_SJDIV   = 'sj_div' in TABLE_COLUMNS
print(f'\ncompany_name={HAS_COMPANY} / market={HAS_MARKET} / sj_div={HAS_SJDIV}')

# ---- 성능 인덱스 권고 ----
# PK(date, ticker, item_code) 만으로는 "항목코드 기준 전체 티커 조회"가 풀스캔이 된다.
# 아래 두 인덱스를 만들어두면 스크리닝 쿼리 속도가 크게 개선된다. (한 번만 실행)
IDX_SQL = [
    f'CREATE INDEX idx_item_date   ON {TABLE_NAME} (item_code, date)',
    f'CREATE INDEX idx_ticker_date ON {TABLE_NAME} (ticker, date)',
]
print('\n[권장 인덱스] 아직 없다면 아래를 1회 실행하세요:')
for s in IDX_SQL:
    name = s.split(' ON ')[0].split()[-1]
    mark = '이미 존재' if name in EXISTING_INDEX else '미생성'
    print(f'  [{mark}] {s};')

# 실행하려면 아래 주석 해제 (370만 rows 기준 수십 초 소요)
# with ENGINE.begin() as conn:
#     for s in IDX_SQL:
#         name = s.split(' ON ')[0].split()[-1]
#         if name not in EXISTING_INDEX:
#             conn.execute(text(s)); print(f'created: {name}')

[스키마] korea_fs_data_from_DG
       Field         Type Null Key             Default                         Extra
        date         date   NO PRI                None                              
      ticker  varchar(10)   NO PRI                None                              
company_name varchar(200)  YES                    None                              
   item_code  varchar(20)   NO PRI                None                              
   indicator varchar(200)  YES MUL                None                              
      sj_div  varchar(10)  YES                    None                              
      market  varchar(10)  YES MUL                None                              
       value       double  YES                    None                              
        freq  varchar(10)  YES                    None                              
  created_at    timestamp  YES     current_timestamp()                              
  updated_at    timestamp  YES     cu

In [11]:
# ==========================================================
# Cell 4: 재무항목(finance item) 마스터 구축 · 검색 함수   ◀ 요구사항 2
# ==========================================================
# 3.7M rows 를 매번 DISTINCT 하면 느리므로, 항목 마스터를 1회 만들어 pickle 로 캐시한다.

ITEM_MASTER_PATH = CACHE_DIR / 'korea_fs_item_master.pkl'


def build_item_master(refresh: bool = False) -> pd.DataFrame:
    """
    DB에 저장된 재무항목 목록을 집계해 반환한다.
    반환 컬럼: item_code, indicator, sj_div, market, row_cnt, ticker_cnt, min_date, max_date
    """
    if (not refresh) and ITEM_MASTER_PATH.exists():
        df = pd.read_pickle(ITEM_MASTER_PATH)
        print(f'항목 마스터 캐시 로드: {ITEM_MASTER_PATH.name}  ({len(df):,} rows)')
        return df

    sel = ['item_code', 'indicator']
    if HAS_SJDIV:
        sel.append('sj_div')
    if HAS_MARKET:
        sel.append('market')
    grp = ', '.join(sel)

    sql = f"""
        SELECT {grp},
               COUNT(*)                AS row_cnt,
               COUNT(DISTINCT ticker)  AS ticker_cnt,
               MIN(date)               AS min_date,
               MAX(date)               AS max_date
        FROM {TABLE_NAME}
        GROUP BY {grp}
    """
    print('항목 마스터 생성 중... (전체 스캔, 수십 초 소요)')
    with ENGINE.connect() as conn:
        df = pd.read_sql(text(sql), conn)

    df = df.sort_values(['row_cnt'], ascending=False).reset_index(drop=True)
    df.to_pickle(ITEM_MASTER_PATH)
    print(f'항목 마스터 저장 완료: {ITEM_MASTER_PATH}  ({len(df):,} rows)')
    return df


def search_items(keyword: Optional[str] = None,
                 sj_div: Optional[str] = None,
                 market: Optional[str] = None,
                 item_code: Optional[str] = None,
                 topn: int = 50) -> pd.DataFrame:
    """
    항목명 부분일치 검색. keyword=None 이면 전체 목록.
    예) search_items('영업이익'), search_items(sj_div='손익계산서')
    """
    df = ITEM_MASTER.copy()
    if keyword:
        df = df[df['indicator'].astype(str).str.contains(keyword, case=False, na=False)]
    if sj_div and 'sj_div' in df.columns:
        df = df[df['sj_div'].astype(str).str.contains(sj_div, case=False, na=False)]
    if market and 'market' in df.columns:
        df = df[df['market'] == market]
    if item_code:
        df = df[df['item_code'].astype(str).str.contains(item_code, case=False, na=False)]
    return df.head(topn).reset_index(drop=True)


def resolve_item_codes(item_names: Optional[List[str]]) -> List[str]:
    """항목명 리스트 → 항목코드 리스트. 미매칭/다중매칭은 경고로 알린다."""
    if not item_names:
        return []
    codes: List[str] = []
    for nm in item_names:
        hit = ITEM_MASTER[ITEM_MASTER['indicator'].astype(str).str.strip() == str(nm).strip()]
        if hit.empty:
            near = ITEM_MASTER[ITEM_MASTER['indicator'].astype(str).str.contains(str(nm), na=False)]
            print(f"  [경고] '{nm}' 정확일치 항목 없음. 유사후보 {len(near)}건:")
            print('        ' + ', '.join(near['indicator'].astype(str).unique()[:8].tolist()))
            continue
        uniq = hit['item_code'].unique().tolist()
        if len(uniq) > 1:
            print(f"  [주의] '{nm}' 에 항목코드가 {len(uniq)}개 매핑됨 → 전부 사용: {uniq}")
        codes.extend(uniq)
    return sorted(set(codes))


# ---- 비금액(비율/주당) 항목 판별: 단위 환산 제외 대상 ----
NON_MONETARY_PAT = (
    r'(?:%|％|비율|률|율|배수|배\)|\(배|회전|주당|EPS|BPS|DPS|SPS|CPS|PER|PBR|PSR|PCR|'
    r'ROE|ROA|ROIC|EV/|배당성향|마진|승수|지수|개월|건수|주식수|인원|종업원|직원|'
    r'등급|점수|일수|횟수)'
)


def is_monetary(indicator: str) -> bool:
    """True면 금액 항목(단위 환산 대상). 비율·주당지표는 False."""
    if indicator is None:
        return True
    return not bool(pd.Series([str(indicator)]).str.contains(NON_MONETARY_PAT, regex=True, na=False).iloc[0])


ITEM_MASTER = build_item_master(refresh=REFRESH_ITEM_MASTER)
print(f'\n총 항목 수 : {ITEM_MASTER["item_code"].nunique():,} 개 코드 / '
      f'{ITEM_MASTER["indicator"].nunique():,} 개 항목명')

항목 마스터 캐시 로드: korea_fs_item_master.pkl  (82 rows)

총 항목 수 : 41 개 코드 / 39 개 항목명


In [12]:
# ==========================================================
# Cell 5: ▶ 항목 검색 실행
# ==========================================================
print(f"[키워드 '{SEARCH_KEYWORD}' 검색결과]")
res = search_items(SEARCH_KEYWORD, topn=ITEM_MASTER_TOPN)
display(res)

# 재무제표 구분별 항목 수
if 'sj_div' in ITEM_MASTER.columns:
    print('\n[sj_div 별 항목 수]')
    print(ITEM_MASTER.groupby('sj_div')['item_code'].nunique()
          .sort_values(ascending=False).to_string())

# 비금액 항목 미리보기 (단위 환산에서 제외될 항목들)
_nm = ITEM_MASTER[~ITEM_MASTER['indicator'].map(is_monetary)]['indicator'].unique()
print(f'\n[비금액 항목 감지] {len(_nm)}개 — 단위 환산 제외 대상')
print('  ' + ', '.join(map(str, _nm[:25])) + (' ...' if len(_nm) > 25 else ''))

[키워드 '매출' 검색결과]


,item_code,indicator,sj_div,market,row_cnt,ticker_cnt,min_date,max_date
0,M000905001,매출원가(천원),IS,KS,46171,788,2009-12-30,2026-05-26
1,M000904001,매출액(천원),IS,KS,45262,788,2009-12-30,2026-03-31
2,M000904007,매출총이익(천원),IS,KS,45257,788,2009-12-30,2026-03-31
3,M001180890,매출채권(**)(천원),BS,KS,43906,777,2009-12-30,2026-03-31
4,M001180890,매출채권(**)(천원),BS,KQ,35205,794,2009-12-30,2026-03-31
5,M000905001,매출원가(천원),IS,KQ,18689,797,2019-12-30,2026-05-29
6,M000904001,매출액(천원),IS,KQ,17221,794,2019-12-30,2026-03-31
7,M000904007,매출총이익(천원),IS,KQ,17219,794,2019-12-30,2026-03-31



[sj_div 별 항목 수]
sj_div
BS       20
IS       11
CF        8
stock     2

[비금액 항목 감지] 2개 — 단위 환산 제외 대상
  평균발행주식수(보통주)(주), 평균발행주식수(보통주,자사주차감)(주)


In [13]:
# ==========================================================
# Cell 6: 재무데이터 추출 함수   ◀ 요구사항 3
# ==========================================================

def _in_clause(col: str, values: List[str], prefix: str, params: Dict) -> str:
    """SQLAlchemy named parameter 방식 IN 절 생성 (문자열 포매팅 주입 방지)."""
    keys = []
    for i, v in enumerate(values):
        k = f'{prefix}{i}'
        params[k] = v
        keys.append(f':{k}')
    return f"{col} IN ({', '.join(keys)})"


def fetch_fs(tickers: Optional[List[str]] = None,
             item_names: Optional[List[str]] = None,
             item_codes: Optional[List[str]] = None,
             start: Optional[str] = None,
             end: Optional[str] = None,
             market: Optional[str] = None,
             sj_div: Optional[str] = None,
             unit: str = '억원',
             verbose: bool = True) -> pd.DataFrame:
    """
    재무항목 + 기간으로 데이터를 추출한다 (long format).

    반환 컬럼: date, ticker, company_name, item_code, indicator, sj_div, market,
              value_raw(천원), value(환산), unit, is_monetary
    """
    codes = list(item_codes) if item_codes else resolve_item_codes(item_names)
    if not codes:
        raise ValueError('조회할 item_code 가 없습니다. ITEM_NAMES / ITEM_CODES 를 확인하세요.')

    cols = ['date', 'ticker', 'item_code', 'indicator', 'value']
    if HAS_COMPANY:
        cols.insert(2, 'company_name')
    if HAS_SJDIV:
        cols.append('sj_div')
    if HAS_MARKET:
        cols.append('market')

    where, params = [], {}
    where.append(_in_clause('item_code', codes, 'ic', params))
    if tickers:
        where.append(_in_clause('ticker', list(tickers), 'tk', params))
    if start:
        where.append('date >= :start'); params['start'] = start
    if end:
        where.append('date <= :end');   params['end'] = end
    if market and HAS_MARKET:
        where.append('market = :market'); params['market'] = market
    if sj_div and HAS_SJDIV:
        where.append('sj_div = :sj_div'); params['sj_div'] = sj_div

    sql = f"""
        SELECT {', '.join(cols)}
        FROM {TABLE_NAME}
        WHERE {' AND '.join(where)}
        ORDER BY ticker, item_code, date
    """
    with ENGINE.connect() as conn:
        df = pd.read_sql(text(sql), conn, params=params)

    if df.empty:
        print('[경고] 조회 결과가 없습니다. 티커/기간/항목 조건을 확인하세요.')
        return df

    df['date'] = pd.to_datetime(df['date'])
    df = df.rename(columns={'value': 'value_raw'})
    df['value_raw'] = pd.to_numeric(df['value_raw'], errors='coerce')

    # ---- 단위 환산: 금액 항목만 (비율·주당지표는 원본 유지) ----
    df['is_monetary'] = df['indicator'].map(is_monetary)
    div = UNIT_DIVISOR[unit]
    df['value'] = np.where(df['is_monetary'], df['value_raw'] / div, df['value_raw'])
    df['unit']  = np.where(df['is_monetary'], unit, '원본')

    if verbose:
        print(f'추출 완료: {len(df):,} rows | '
              f'{df["ticker"].nunique()} 종목 | {df["indicator"].nunique()} 항목 | '
              f'{df["date"].min().date()} ~ {df["date"].max().date()}')
        nonmon = df.loc[~df['is_monetary'], 'indicator'].unique()
        if len(nonmon):
            print(f'  · 단위 미환산(비금액) 항목: {", ".join(map(str, nonmon))}')
    return df


def to_wide(df: pd.DataFrame,
            value_col: str = 'value',
            index: str = 'date',
            columns: Union[str, List[str]] = 'indicator') -> pd.DataFrame:
    """long → wide 피벗. 종목이 여러 개면 columns=['ticker','indicator'] 권장."""
    return df.pivot_table(index=index, columns=columns, values=value_col, aggfunc='last').sort_index()


def detect_freq(df: pd.DataFrame) -> str:
    """날짜 간격으로 데이터 주기 판정: 'Q'(분기) 또는 'A'(연간)."""
    d = pd.Series(sorted(df['date'].unique()))
    if len(d) < 3:
        return 'A'
    gap = pd.Series(d).diff().dt.days.median()
    return 'Q' if gap is not None and gap < 200 else 'A'

In [14]:
# ==========================================================
# Cell 7: ▶ 데이터 추출 실행
# ==========================================================
df_fs = fetch_fs(
    tickers    = TICKERS,
    item_names = ITEM_NAMES,
    item_codes = ITEM_CODES,
    start      = START_DATE,
    end        = END_DATE,
    market     = MARKET,
    sj_div     = SJ_DIV,
    unit       = UNIT,
)

# 데이터 주기 확정
FREQ_USED = detect_freq(df_fs) if FREQ == 'auto' else FREQ
LAG       = GROWTH_LAG if GROWTH_LAG else (4 if FREQ_USED == 'Q' else 1)
print(f'\n데이터 주기: {FREQ_USED}  |  성장률 lag: {LAG}')

display(df_fs.tail(10))

# wide 미리보기 (종목 × 항목)
if not df_fs.empty:
    print('\n[wide 피벗 미리보기]')
    display(to_wide(df_fs, columns=['ticker', 'indicator']).tail(8))

추출 완료: 156 rows | 2 종목 | 2 항목 | 2015-03-31 ~ 2026-03-31

데이터 주기: Q  |  성장률 lag: 4


,date,ticker,company_name,item_code,indicator,value_raw,sj_div,market,is_monetary,value,unit
146,2023-12-28,A278470,에이피알,M000906001,영업이익(천원),"34,359,650.87",IS,KS,True,343.60,억원
147,2024-03-29,A278470,에이피알,M000906001,영업이익(천원),"27,765,383.70",IS,KS,True,277.65,억원
148,2024-06-28,A278470,에이피알,M000906001,영업이익(천원),"28,011,311.86",IS,KS,True,280.11,억원
149,2024-09-30,A278470,에이피알,M000906001,영업이익(천원),"27,243,326.88",IS,KS,True,272.43,억원
150,2024-12-30,A278470,에이피알,M000906001,영업이익(천원),"39,685,521.99",IS,KS,True,396.86,억원
151,2025-03-31,A278470,에이피알,M000906001,영업이익(천원),"54,568,141.91",IS,KS,True,545.68,억원
152,2025-06-30,A278470,에이피알,M000906001,영업이익(천원),"84,552,442.45",IS,KS,True,845.52,억원
153,2025-09-30,A278470,에이피알,M000906001,영업이익(천원),"96,127,667.28",IS,KS,True,961.28,억원
154,2025-12-30,A278470,에이피알,M000906001,영업이익(천원),"130,272,377.30",IS,KS,True,"1,302.72",억원
155,2026-03-31,A278470,에이피알,M000906001,영업이익(천원),"145,567,469.86",IS,KS,True,"1,455.67",억원



[wide 피벗 미리보기]


ticker     A003350           A278470         
indicator  매출액(천원) 영업이익(천원)  매출액(천원) 영업이익(천원)
date                                         
2024-06-28  481.14    89.49 1,554.94   280.11
2024-09-30  482.23    93.61 1,741.17   272.43
2024-12-30  359.97    38.74 2,442.15   396.86
2025-03-31  379.98    60.01 2,660.33   545.68
2025-06-30  537.74   108.25 3,277.35   845.52
2025-09-30  548.36   118.53 3,859.43   961.28
2025-12-30  380.20    42.05 5,476.35 1,302.72
2026-03-31  522.27    85.75 5,933.56 1,455.67

In [17]:
df_fs.to_csv(r"C:\Users\82108\OneDrive\INVESTMENT\한국주식\에이피알\kscm_fs_data.csv")

In [15]:
# ==========================================================
# Cell 8: 기본 분석 (YoY · QoQ · TTM · CAGR)   ◀ 요구사항 5-1
# ==========================================================

def add_growth(df: pd.DataFrame,
               lag: int = 4,
               add_ttm: bool = True,
               freq: str = 'Q') -> pd.DataFrame:
    """
    종목×항목 단위로 성장률 지표를 추가한다.
      yoy    : lag 기 대비 증감률(%)
      qoq    : 직전기 대비 증감률(%)
      ttm    : 4분기 누적 (분기 유량항목만)
      ttm_yoy: TTM 기준 전년동기 대비 증감률(%)
    부호가 뒤집히는 구간(적자→흑자 등)은 성장률이 왜곡되므로 NaN 처리한다.
    """
    out = []
    for (tk, code), g in df.sort_values('date').groupby(['ticker', 'item_code'], sort=False):
        g = g.copy()
        v = g['value']

        prev_lag = v.shift(lag)
        prev_1   = v.shift(1)

        # 기준값이 0 이하이면 증감률 해석 불가 → NaN
        g['yoy'] = np.where(prev_lag > 0, (v / prev_lag - 1) * 100, np.nan)
        g['qoq'] = np.where(prev_1  > 0, (v / prev_1  - 1) * 100, np.nan)

        if add_ttm and freq == 'Q' and bool(g['is_monetary'].iloc[0]):
            ttm = v.rolling(4, min_periods=4).sum()
            g['ttm'] = ttm
            prev_ttm = ttm.shift(4)
            g['ttm_yoy'] = np.where(prev_ttm > 0, (ttm / prev_ttm - 1) * 100, np.nan)
        else:
            g['ttm'] = np.nan
            g['ttm_yoy'] = np.nan

        out.append(g)

    res = pd.concat(out, ignore_index=True) if out else df.copy()
    return res.sort_values(['ticker', 'item_code', 'date']).reset_index(drop=True)


def summarize(df: pd.DataFrame, freq: str = 'Q') -> pd.DataFrame:
    """종목×항목별 요약통계: 최근값 · CAGR · 평균/중앙 성장률 · 변동성."""
    periods_per_year = 4 if freq == 'Q' else 1
    rows = []
    for (tk, code), g in df.groupby(['ticker', 'item_code'], sort=False):
        g = g.sort_values('date').dropna(subset=['value'])
        if g.empty:
            continue
        first, last = g['value'].iloc[0], g['value'].iloc[-1]
        n_years = (len(g) - 1) / periods_per_year

        cagr = np.nan
        if n_years > 0 and first > 0 and last > 0:
            cagr = ((last / first) ** (1 / n_years) - 1) * 100

        rows.append({
            'ticker':     tk,
            'company':    g['company_name'].iloc[-1] if 'company_name' in g.columns else '',
            'indicator':  g['indicator'].iloc[-1],
            'item_code':  code,
            'unit':       g['unit'].iloc[-1],
            'n_periods':  len(g),
            'first_date': g['date'].iloc[0].date(),
            'last_date':  g['date'].iloc[-1].date(),
            'first':      first,
            'last':       last,
            'CAGR(%)':    cagr,
            'YoY_최근(%)': g['yoy'].iloc[-1] if 'yoy' in g else np.nan,
            'YoY_평균(%)': g['yoy'].mean() if 'yoy' in g else np.nan,
            'YoY_중앙(%)': g['yoy'].median() if 'yoy' in g else np.nan,
            'YoY_표준편차': g['yoy'].std() if 'yoy' in g else np.nan,
        })
    return pd.DataFrame(rows)


df_ana = add_growth(df_fs, lag=LAG, add_ttm=ADD_TTM, freq=FREQ_USED)
df_sum = summarize(df_ana, freq=FREQ_USED)

print('[요약 통계]')
display(df_sum)

print('\n[최근 8기 상세]')
_cols = ['date', 'ticker', 'indicator', 'value', 'unit', 'yoy', 'qoq', 'ttm', 'ttm_yoy']
_cols = [c for c in _cols if c in df_ana.columns]
display(df_ana.groupby(['ticker', 'item_code']).tail(8)[_cols])

[요약 통계]


,ticker,company,indicator,item_code,unit,n_periods,first_date,last_date,first,last,CAGR(%),YoY_최근(%),YoY_평균(%),YoY_중앙(%),YoY_표준편차
0,A003350,한국화장품제조,매출액(천원),M000904001,억원,45,2015-03-31,2026-03-31,82.55,522.27,18.26,37.45,19.10,17.26,29.21
1,A003350,한국화장품제조,영업이익(천원),M000906001,억원,45,2015-03-31,2026-03-31,3.75,85.75,32.91,42.89,-60.16,18.52,508.30
2,A278470,에이피알,매출액(천원),M000904001,억원,33,2018-03-30,2026-03-31,204.37,"5,933.56",52.36,123.04,57.98,57.37,40.15
3,A278470,에이피알,영업이익(천원),M000906001,억원,33,2018-03-30,2026-03-31,24.77,"1,455.67",66.40,166.76,149.13,114.55,189.95



[최근 8기 상세]


,date,ticker,indicator,value,unit,yoy,qoq,ttm,ttm_yoy
37,2024-06-28,A003350,매출액(천원),481.14,억원,73.25,36.94,"1,379.07",39.26
38,2024-09-30,A003350,매출액(천원),482.23,억원,71.18,0.23,"1,579.58",53.09
39,2024-12-30,A003350,매출액(천원),359.97,억원,35.91,-25.35,"1,674.69",53.17
40,2025-03-31,A003350,매출액(천원),379.98,억원,8.15,5.56,"1,703.32",44.88
41,2025-06-30,A003350,매출액(천원),537.74,억원,11.76,41.52,"1,759.92",27.62
42,2025-09-30,A003350,매출액(천원),548.36,억원,13.71,1.98,"1,826.06",15.60
43,2025-12-30,A003350,매출액(천원),380.20,억원,5.62,-30.67,"1,846.28",10.25
44,2026-03-31,A003350,매출액(천원),522.27,억원,37.45,37.37,"1,988.57",16.75
82,2024-06-28,A003350,영업이익(천원),89.49,억원,255.20,105.22,176.12,232.13
83,2024-09-30,A003350,영업이익(천원),93.61,억원,243.96,4.61,242.51,235.35


In [ ]:
# ==========================================================
# Cell 9: 시각화 함수   ◀ 요구사항 5-2
# ==========================================================

def _label(g: pd.DataFrame) -> str:
    nm = g['company_name'].iloc[-1] if 'company_name' in g.columns and pd.notna(g['company_name'].iloc[-1]) else ''
    return f"{nm}({g['ticker'].iloc[-1]})" if nm else str(g['ticker'].iloc[-1])


def plot_item(df: pd.DataFrame,
              indicator: str,
              freq: str = 'Q',
              figwidth: int = 15) -> None:
    """단일 재무항목에 대해 4패널 분석 차트를 그린다."""
    d = df[df['indicator'] == indicator].copy()
    if d.empty:
        print(f'[스킵] 데이터 없음: {indicator}')
        return

    unit = d['unit'].iloc[-1]
    has_ttm = d['ttm'].notna().any()

    fig, axes = plt.subplots(2, 2, figsize=(figwidth, 8.5))
    fig.suptitle(f'{indicator}  ({unit} / {"분기" if freq == "Q" else "연간"})',
                 fontsize=15, fontweight='bold')

    # (1) 레벨 추이
    ax = axes[0, 0]
    for tk, g in d.groupby('ticker', sort=False):
        g = g.sort_values('date')
        ax.plot(g['date'], g['value'], marker='o', ms=3.5, lw=1.6, label=_label(g))
    ax.set_title('추이 (Level)', fontsize=11)
    ax.set_ylabel(unit); ax.grid(alpha=0.3); ax.legend(fontsize=9)
    ax.axhline(0, color='grey', lw=0.8)

    # (2) YoY 성장률
    ax = axes[0, 1]
    tickers = list(d['ticker'].unique())
    width = 0.8 / max(len(tickers), 1)
    for i, tk in enumerate(tickers):
        g = d[d['ticker'] == tk].sort_values('date')
        x = np.arange(len(g)) + i * width
        ax.bar(x, g['yoy'], width=width, label=_label(g), alpha=0.85)
        if i == 0:
            step = max(1, len(g) // 10)
            ax.set_xticks(np.arange(len(g))[::step])
            ax.set_xticklabels([dt.strftime('%y-%m') for dt in g['date']][::step],
                               rotation=45, ha='right', fontsize=8)
    ax.axhline(0, color='black', lw=0.9)
    ax.set_title(f'YoY 성장률 (lag={LAG})', fontsize=11)
    ax.set_ylabel('%'); ax.grid(alpha=0.3, axis='y'); ax.legend(fontsize=9)

    # (3) TTM 또는 QoQ
    ax = axes[1, 0]
    if has_ttm:
        for tk, g in d.groupby('ticker', sort=False):
            g = g.sort_values('date')
            ax.plot(g['date'], g['ttm'], marker='o', ms=3, lw=1.6, label=_label(g))
        ax.set_title('TTM (4분기 누적)', fontsize=11); ax.set_ylabel(unit)
    else:
        for tk, g in d.groupby('ticker', sort=False):
            g = g.sort_values('date')
            ax.plot(g['date'], g['qoq'], marker='o', ms=3, lw=1.4, label=_label(g))
        ax.axhline(0, color='black', lw=0.9)
        ax.set_title('직전기 대비 증감률 (QoQ)', fontsize=11); ax.set_ylabel('%')
    ax.grid(alpha=0.3); ax.legend(fontsize=9)

    # (4) YoY 분포 (박스플롯)
    ax = axes[1, 1]
    data, labels = [], []
    for tk, g in d.groupby('ticker', sort=False):
        vals = g['yoy'].dropna().values
        if len(vals):
            data.append(vals); labels.append(_label(g))
    if data:
        try:                      # matplotlib >= 3.9
            bp = ax.boxplot(data, tick_labels=labels, patch_artist=True, showmeans=True)
        except TypeError:         # matplotlib < 3.9
            bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True)
        for patch in bp['boxes']:
            patch.set_alpha(0.55)
        ax.axhline(0, color='black', lw=0.9)
    ax.set_title('YoY 분포 (변동성)', fontsize=11)
    ax.set_ylabel('%'); ax.grid(alpha=0.3, axis='y')
    plt.setp(ax.get_xticklabels(), rotation=15, ha='right', fontsize=9)

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()


def plot_all(df: pd.DataFrame, freq: str = 'Q', figwidth: int = 15) -> None:
    """추출된 모든 항목에 대해 순차적으로 차트를 그린다."""
    for ind in df['indicator'].dropna().unique():
        plot_item(df, ind, freq=freq, figwidth=figwidth)


def plot_compare(df: pd.DataFrame, indicator: str, base_index: bool = True,
                 figwidth: int = 15) -> None:
    """여러 종목의 동일 항목을 100 기준 지수화해 비교한다."""
    d = df[df['indicator'] == indicator].copy()
    if d.empty:
        print(f'[스킵] 데이터 없음: {indicator}'); return

    fig, ax = plt.subplots(figsize=(figwidth, 4.5))
    for tk, g in d.groupby('ticker', sort=False):
        g = g.sort_values('date').dropna(subset=['value'])
        if g.empty:
            continue
        y = g['value'] / g['value'].iloc[0] * 100 if base_index and g['value'].iloc[0] != 0 else g['value']
        ax.plot(g['date'], y, marker='o', ms=3, lw=1.8, label=_label(g))
    ax.set_title(f'{indicator} — ' + ('시작=100 지수화 비교' if base_index else '절대값 비교'),
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Index (시작=100)' if base_index else d['unit'].iloc[-1])
    ax.grid(alpha=0.3); ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()

In [ ]:
# ==========================================================
# Cell 10: ▶ 시각화 실행
# ==========================================================
plot_all(df_ana, freq=FREQ_USED, figwidth=FIG_WIDTH)

# 종목 간 상대비교 (첫 번째 항목 기준)
if df_ana['ticker'].nunique() > 1:
    first_item = df_ana['indicator'].dropna().unique()[0]
    plot_compare(df_ana, first_item, base_index=True, figwidth=FIG_WIDTH)

In [ ]:
# ==========================================================
# Cell 11: 엑셀 저장 (옵션)
# ==========================================================
if SAVE_EXCEL:
    from datetime import datetime
    OUT_DIR = PROJECT_ROOT / 'OUTPUT'
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().strftime('%Y%m%d_%H%M')
    out_path = OUT_DIR / f'korea_fs_query_{stamp}.xlsx'

    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
        df_sum.to_excel(writer, sheet_name='요약', index=False)
        df_ana.to_excel(writer, sheet_name='상세(long)', index=False)
        to_wide(df_ana, columns=['ticker', 'indicator']).to_excel(writer, sheet_name='wide')
        search_items(SEARCH_KEYWORD, topn=500).to_excel(writer, sheet_name='항목검색', index=False)

    print(f'저장 완료: {out_path}')
else:
    print('SAVE_EXCEL=False → 저장 건너뜀')